<a href="https://colab.research.google.com/github/Viggo-Kristensen/kaggle-competitions/blob/main/irrigation_water_need.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1. Problem Definition**

### Predict the amount of irrigation required under varying environmental conditions

## **2. Imports**

In [1]:
import pandas as pd
import numpy as np
import os


# Pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split

# Metrics
from sklearn.metrics import balanced_accuracy_score

## **3. Dataset**

Each row in the dataset is a unique datapoint storing information for a specific crop. The target variable is `Irrigation_Need` and indicates how much water the plant needs. The possible values for the target are "Low", "Medium" and "High".










### Uploading files

In [2]:
!mkdir -p /root/.kaggle
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [3]:
# upload kaggle.json ONCE
from google.colab import files
files.upload()

!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c playground-series-s6e4

Saving kaggle.json to kaggle.json
100% 32.5M/32.5M [00:02<00:00, 12.9MB/s]



### Unzipping files

In [4]:
!unzip -q playground-series-s6e4.zip -d data

### Creating Dataframes

In [5]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

In [6]:
target = "Irrigation_Need"

drop_cols = [target, "id"]

X = train.drop(columns=drop_cols)
y = train[target]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "bool"]).columns

## **4. Feature Engineering**

In [7]:
# Custom feature engineering based on agronomic intuition and environmental relationships
class AddFeatures(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self

  def transform(self, X):
    X = X.copy()

    # Estimate water stress by comparing soil moisture against rainfall
    X["Water_deficit"] = X["Soil_Moisture"] - X["Rainfall_mm"]

    # Ratios and normalized quantities
    X["Temperature_Humidity_Ratio"] = X["Temperature_C"] / (X["Humidity"] + 1e-6)

    return X


## **5. Preprocessing**

### Numerical transformer

In [8]:
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

### Categorical transformer

In [9]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

### preprocessor

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ],
    remainder="passthrough"
)

## **6. Pipeline**

In [11]:
model = Pipeline(steps=[
    ("feature_engineering", AddFeatures()),
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=30,
        max_depth=21,
        min_samples_leaf=5,
        min_samples_split=10,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )
    )
])

## **7. Cross Validation**

In [12]:
scores = cross_val_score(model, X, y, cv=5, scoring="balanced_accuracy", n_jobs=-1)

print("scores:", scores)
print("mean score:", scores.mean())

scores: [0.96076284 0.95904693 0.96100555 0.9616588  0.95613663]
mean score: 0.9597221502917286


## **8. Training**

In [13]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('feature_engineering', AddFeatures()),
                ('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  Index(['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
       'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
       'Wind_Speed_kmh', 'Fi...
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
       'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region'],
      dtype='object'))])),
                ('classifier',
                 RandomForestClassifier(max_depth=21, min_samples_leaf=5,
                                        min_samples_split=10, n_estimators=30,
                                        n_jobs=-1, random_state=42))])

## **9. Evaluation**

In [14]:
y_preds = model.predict(X_val)
score = balanced_accuracy_score(y_val, y_preds)
print(score)

0.9576249496204542


## **10. Submission**

In [15]:
submission = pd.read_csv("data/sample_submission.csv")
submission["Irrigation_Need"] = model.predict(test)
submission.to_csv("submission.csv", index=False)

## **11. Reflections**

**Notes**
*   21 was the choice for `max_depth` that lead to the highest accuracy score for the random forest
*  `Water_Deficit` was the best engineered feature

**What i have learned**
*   Which feature engineering choices are good for decision trees
*   How bias and variance are different in random forests and single decision trees